<a href="https://colab.research.google.com/github/ablasve/Mini-Proyecto-Asistente-Multimodal-de-Salud/blob/main/FaseA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase A: Gestión del Asistente

En esta primera fase del proyecto definiremos a grandes rasgos la estructura global del funcionamiento del asistente: las opciones que dará, la memoria a la que tendrá acceso, la estructura de esta última, etc. Nuestros principales objetivos serán simular una interfaz clara y ordenada, lo más sencilla e intuitiva posible para que el asistente sea accesible y fácil de usar.

**Contenidos** <a id='toc0_'></a>   

[Librerías necesarias](#toc1)

[Esquema global del funcionamiento del asistente](#toc2)

[Interfaz del asistente](#toc3)

- [Saludo inicial](toc3_1)
  - [Bloque de memoria](toc3_1_1)
  - [Sin memoria previa: presentación y recabación de datos](toc3_1_2)
  - [Con memoria previa: saludo breve](toc3_1_3)

- [Menú de opciones](toc3_1)

- [Cohesión: función global](toc3_1)


## <a id='toc1'></a>[0. Librerías necesarias](#toc0_)


In [ ]:
# Para administrar ARCHIVOS:
import json
import os
from google.colab import files

# Para administrar IMÁGENES:
import PIL.Image

# Para grabar AUDIO:
!pip install openai-whisper -q
import whisper
from google.colab import output
from base64 import b64decode

# Para reproducir AUDIO:
!pip install edge-tts -q
import asyncio
import edge_tts
from IPython.display import Audio, display


# Para pasar de VOZ a AUDIO:
from whisper import load_model
model_whisper = load_model("small") # Cargamos el modelo Whisper

In [ ]:
!pip install -q bitsandbytes
from transformers import BitsAndBytesConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
torch.cuda.empty_cache()

# Cargamos el modelo de Texto puro para MIRAR SI LO VAMOS A USAR O NO!!!!!
#id_model_texto = "Qwen/Qwen2.5-1.5B-Instruct"
id_model_texto = "Qwen/Qwen2.5-3B-Instruct"

# Configuración para ahorrar memoria (4 bits)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32
)


tokenizer_texto = AutoTokenizer.from_pretrained(id_model_texto)
model_texto = AutoModelForCausalLM.from_pretrained(
    id_model_texto,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Modelo de TEXTO cargado con éxito")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

## <a id='toc2'></a>[1. Esquema global del funcionamiento del asistente](#toc0_)

Debido a la naturaleza de este mini-proyecto, vamos a enfocar el asistente en un único usuario, por lo que solo guardaremos información de dicho usuario. Cuando se empiece una consulta al asistente, este buscará en su "memoria" (datos proporcionados en consultas previas, que se guardaran en un archivo con formato _JSON_):
- si no encuentra este archivo, creará uno vacío, se presentará y le pedirá al usuario un nombre por el que poder dirigirse a él/ella. Después de recabar el nombre, procederá a ofrecerle los servicios que abarca (profundizado en Fase B).
- si encuentra el archivo, sabrá el nombre del usuario, por lo que directamente le saludará y ofrecerá distintas opciones.

A parte de las opciones de asistencia en el ámbito de salud (introducción de medicamentos en el registro, respuesta a preguntas sencillas y resumen del historial de medicinas), también ofreceremos otras opciones técnicas pertinentes, como el cambio de nombre, la supresión parcial o total de la información recabada en el registro, etc. Sin embargo, para simplificar el esquema se podrá acceder a estas opciones mediante la opción de _Ajustes_.

Una vez realizada y finalizada la primera consulta, el asistente preguntará amablemente si se desea continuar hablando con el asistente o si, por el contrario, no desean consultar nada más. En el primer caso, el asistente volverá a ofrecer las opciones disponibles y, en el segundo, dará por finalizada la sesión y se cerrará la aplicación, guardándose toda la información de dicha sesión.

## <a id='toc3'></a>[2. Interfaz del asistente](#toc0_)

### <a id='toc3_1'></a>[2.1. Saludo inicial](#toc0_)

#### <a id='toc3_1_1'></a>[2.1.1. Bloque de memoria](#toc0_)

El primer paso para poder estructurar las interacciones asistente-usuario será crear un histórico con la información proporcionada. Si el archivo de memoria no existe, se entiende que estamos hablando por primera vez con el usuario.

Como la primera consulta y todas las posteriores se basarán en la información recabada en la memoria, es importante fijar una estructura clara y precisa para el documento. Por ello, hemos decidido que el archivo tendrá formato _JSON_ y contará con los siguientes campos:

- _nombre_ : apelativo con el que el asistente interpelará al usuario.
- _medicinas_ : lista de diccionarios que contendrá el nombre, dosis, frecuencia, fechas inicial y final de los medicamentos.
- _ultimas_adiciones_ : lista con los nombres de los últimos medicamentos agregados a la lista de _medicinas_ (este último apartado será útil a la hora de registrar nuevos medicamentos vía imágenes).

Pasando de la teoría a la práctica hemos escrito la siguiente función, que busca el archivo de memoria y, si no lo encuentra, crea uno vacío:

In [ ]:
def cargar_memoria():
    if os.path.exists("memoria_salud.json"):
        with open("memoria_salud.json", "r") as f:
            return json.load(f)
    else:
        # Si no existe, creamos un perfil vacío
        return {"nombre": None, "medicinas": [], "historial": []}

In [ ]:
def guardar_memoria(datos):
    with open("memoria_salud.json", "w") as f:
        json.dump(datos, f, indent=4)

In [ ]:
# Inicializamos la memoria al empezar
memoria = cargar_memoria()

#### <a id='toc3_1_2'>[2.1.2 Sin memoria previa: presentación y recabación de datos](#toc0_)

En este apartado abordamos el primer contacto entre el usuario y el asistente. Cuando nuestra función de carga determina que no existe un historial previo (es decir, el campo "nombre" se encuentra vacío o es None), el sistema asume que es la primera vez que interactúa con la persona. Por tanto, el objetivo aquí es programar una rutina de bienvenida donde el asistente se presente y solicite el nombre del usuario para poder ofrecerle un trato cercano y personalizado. Una vez recabado este dato mediante un input por pantalla, actualizaremos la memoria para que incluya el nombre de este usuario y así posteriormente poder guardar su historial médico.

Primero el programa solicita el nombre, lo registra en la variable de memoria y, lo más importante, llama a la función guardar_memoria().

Añadimos el modelo de escucha y de transcripción de audio que teníamos en la Fase B.

In [1]:

# Función para generar y reproducir el audio
async def generar_voz(texto):
    # Elegimos la voz: 'es-ES-ElviraNeural' (Mujer, España, muy clara)
    # O 'es-ES-AlvaroNeural' si prefieres hombre.
    VOICE = "es-ES-ElviraNeural"
    OUTPUT_FILE = "respuesta_asistente.mp3"

    communicate = edge_tts.Communicate(texto, VOICE, rate="-10%") # rate="-10%" si la quieres más lenta
    await communicate.save(OUTPUT_FILE)

    # Reproducir en Colab
    display(Audio(OUTPUT_FILE, autoplay=True))

# Forma de ejecutar la función:
# await generar_voz(respuesta_texto)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 34.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 7.5 MB/s eta 0:00:00


100%|████████████████████████████████████████| 461M/461M [00:02<00:00, 216MiB/s]


In [2]:
# Código JavaScript para grabar audio desde el navegador
RECORD_JS = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  const recorder = new MediaRecorder(stream)
  const chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    const blob = new Blob(chunks)
    const text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def grabar_audio(segundos=5):
    print(f"Escuchando durante {segundos} segundos...")
    output.eval_js(RECORD_JS)
    audio_b64 = output.eval_js(f"record({segundos*1000})")
    audio_bytes = b64decode(audio_b64.split(',')[1])
    with open("audio_usuario.wav", "wb") as f:
        f.write(audio_bytes)
    return "audio_usuario.wav"

In [ ]:
async def presentacion(memoria):
    if memoria.get("nombre") is None:
        texto_bienvenida = "¡Hola! Soy tu Asistente de Salud. Veo que es nuestra primera vez hablando. Para poder ayudarte mejor... ¿cómo te llamas?"

        print(texto_bienvenida)
        await generar_voz(texto_bienvenida)

        # Pausa para que termine de hablar antes de grabar
        await asyncio.sleep(7)

        # Grabamos al usuario
        print("Escuchando...")
        archivo_wav = grabar_audio(segundos=5)

        # 1. Transcribimos el audio con Whisper
        resultado = model_whisper.transcribe(archivo_wav, language="es")
        texto_bruto = resultado["text"].strip()
        print(f"Has dicho: '{texto_bruto}'")

        # 2. PROCESAMOS EL NOMBRE CON QWEN
        print("Procesando tu nombre...")

        # Preparamos el prompt estricto para extraer solo el nombre
        mensajes = [
            {"role": "system", "content": "Eres un asistente experto en extracción de entidades. Tu única tarea es leer una frase y devolver ÚNICAMENTE el nombre propio de la persona que se presenta. No añadas puntos, ni saludos, ni explicaciones. Solo la palabra del nombre."},
            {"role": "user", "content": f"Extrae el nombre de esta frase: '{texto_bruto}'"}
        ]

        # Tokenizamos y mandamos al modelo
        texto_prompt = tokenizer_texto.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer_texto([texto_prompt], return_tensors="pt").to(model_texto.device)

        # Generamos la respuesta con temperature baja para mayor precisión
        outputs = model_texto.generate(**inputs, max_new_tokens=10, temperature=0.1)
        nombre_limpio = tokenizer_texto.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

        print(f"Nombre extraído: '{nombre_limpio}'")
        # ========================================================

        #Guardamos el nombre limpio en memoria
        memoria["nombre"] = nombre_limpio
        guardar_memoria(memoria)

        texto_confirmacion = f"¡Perfecto, {memoria['nombre']}! He creado tu perfil. Ya podemos empezar con las consultas."
        print(f"\n {texto_confirmacion}")
        await generar_voz(texto_confirmacion)

    else:
        texto_retorno = f"¡Hola de nuevo, {memoria['nombre']}! Tu perfil ya estaba cargado."
        print(f"{texto_retorno}")
        await generar_voz(texto_retorno)

In [ ]:
# Ejecutamos la función
# await presentacion_multimodal(memoria)